In [1]:
import random
import numpy as np
import torch
import pandas as pd
import os
import sys
scripts_path = os.path.abspath('../scripts')
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

from AXIS_tools import *

seed_everything(98)
#worker_init_fn(0, 0, 11)

In [2]:
# %%
from AXIS_model import *
test_fold = [3]
synergy_name = 'zip'
print(f"Test Fold {test_fold[0]} 开始")
import pandas as pd

from transformers import AutoTokenizer
import pickle
import os 
#from torch.utils.data import DataLoader, Datase

# %% [markdown]
# #### 模型定义

# %%




# %%
mlp = Predictor_eval()

# %%
d1_t = torch.ones([4,768]) 
g_t = torch.ones([4,256])
p_t = torch.ones([4,447])
k_t = torch.ones([4,178])
m_t = torch.ones([4,225])
m_tt = torch.ones([4,1024])
geneEffect_t = torch.ones([4,256])
mlp(d1_t,d1_t,g_t,k_t,p_t,m_t,geneEffect_t,geneEffect_t,geneEffect_t,geneEffect_t,g_t,g_t)

Test Fold 3 开始


(tensor([[0.0193],
         [0.0083],
         [0.0265],
         [0.0024]], grad_fn=<DivBackward0>),
 tensor([[0.6707],
         [0.6704],
         [0.6770],
         [0.6844]], grad_fn=<SoftplusBackward0>))

In [3]:
from torch.utils.data import DataLoader, Dataset
# %% [markdown]
# #### 模型训练

# %%
import torch
torch.cuda.is_available()

# %%
device2 = torch.device('cuda:0')
mlp.to(device2)

# %%
#参数
peft_lr = 5e-5
mlp_lr = 1e-5
batch_size = 512
test_batch = 1024
display ='430'
output_dir = f"../models/output"
log_dir = f"{output_dir}/log"

if  not os.path.exists(f"{output_dir}/result_{test_fold[0]}"):
    os.mkdir(f"{output_dir}/result_{test_fold[0]}")

# %%
#from torch.optim import AdamW
import os
from transformers import AdamW, get_linear_schedule_with_warmup,get_cosine_with_hard_restarts_schedule_with_warmup
from transformers import get_cosine_with_hard_restarts_schedule_with_warmup as warmup

#optimizer1 = AdamW(model.parameters(), lr=peft_lr,weight_decay=0.01)
optimizer2 = AdamW(mlp.parameters(), lr=mlp_lr)
from transformers import get_scheduler

from torch.utils.data import DataLoader, SequentialSampler

import torch
#torch.save(train_dataloader0, 'drugcombo_fold_0_data.pt')
# 加载使用 torch.save 保存的数据
# 2. 优化 Dataset 类
class mydata1(Dataset):
    def __init__(self, data):
        self.data = data
        # 预处理数据
        self._preprocess_data()
        
    def _preprocess_data(self):
        # 提前将数据转换为张量并移到CPU内存
        self.processed_data = []
        for item in self.data:
            processed_item = {
                "drug_f1": torch.as_tensor(item['drug_f1'], dtype=torch.float32),
                "drug_f2": torch.as_tensor(item['drug_f2'], dtype=torch.float32),
                "gene_f": torch.as_tensor(item['gene_f'], dtype=torch.float32),
                "protein_f": torch.as_tensor(item['protein_f'], dtype=torch.float32),
                "kegg_f": torch.as_tensor(item['kegg_f'], dtype=torch.float32),
                "meta_f": torch.as_tensor(item['meta_f'], dtype=torch.float32),
                "Bliss": torch.as_tensor([item['synergy']], dtype=torch.float32),
                "geneEffect_f": torch.as_tensor(item['geneEffect_f'], dtype=torch.float32),
                "ssGSEA_f": torch.as_tensor(item['ssGSEA_f'], dtype=torch.float32),
                "geneDependency_f": torch.as_tensor(item['geneDependency_f'], dtype=torch.float32),
                "methylation_f": torch.as_tensor(item['methylation_f'], dtype=torch.float32),
                "CNV_f": torch.as_tensor(item['CNV_f'], dtype=torch.float32),
                "mutation_f": torch.as_tensor(item['mutation_f'], dtype=torch.float32),
            }
            self.processed_data.append(processed_item)

    def __len__(self):
        return len(self.processed_data)
    
    def __getitem__(self, index):
        return self.processed_data[index]



/mnt/hpc/home/shilei/miniconda3/envs/nrf-llama2-2/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [4]:


data_fold_ls = [0,1,2,3,4]
tran_fold = [i for i in data_fold_ls if i not in test_fold]
data_set_dir = "../data/DrugComb"

test_0 = torch.load(f'{data_set_dir}/drugcomb_fold_{test_fold[0]}_data.pt')

from torch.utils.data import ConcatDataset
test_dataloader = DataLoader(
    test_0, 
    batch_size=test_batch)

In [5]:
mlp.load_state_dict(torch.load(f'../models/AXIS.pth'))

# %%
with torch.no_grad():
    #biot5_model.eval()
    mlp.eval()
    L_ls1 = []
    O_ls1 = []
    for batch in test_dataloader:
        drug_f1 = batch["drug_f1"][:,0:768].to(device2)
        drug_f2 = batch["drug_f2"][:,0:768].to(device2)
        Bliss = batch[f"{synergy_name}"][:,0].to(device2)
        gene_f = batch["gene_f"][:,0:256].to(device2)
        kegg_f = batch["kegg_f"][:,0:178].to(device2)
        protein_f = batch["protein_f"][:,0:447].to(device2)
        meta_f = batch["meta_f"][:,0:225].to(device2)
        geneEffect_f = batch["geneEffect_f"].to(device2)
        ssGSEA_f = batch["ssGSEA_f"].to(device2)
        geneDependency_f = batch["geneDependency_f"].to(device2)
        methylation_f = batch["methylation_f"].to(device2)
        CNV_f = batch["CNV_f"].to(device2)
        mutation_f = batch["mutation_f"].to(device2)
        outputs,var = mlp(drug_f1,drug_f2,gene_f,kegg_f, protein_f,meta_f,geneEffect_f,ssGSEA_f, geneDependency_f, methylation_f, CNV_f, mutation_f)
        L_ls1 = L_ls1 + Bliss.cpu().numpy().tolist()
        O_ls1 = O_ls1 + outputs.cpu().numpy().tolist()

O_ls1 = [O[0] for O in O_ls1]

# %%
import numpy as np
x = np.array(L_ls1)
y = np.array(O_ls1)
print("RMSE:",np.sum(np.abs(x-y))/len(y))
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# 假设我们有两组数据
x = np.array(L_ls1)
y = np.array(O_ls1)

# 计算Pearson相关系数
#corr, _ = pearsonr(x, y)
#corr = corr.tolist()[0]
result = pearsonr(x, y)

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# 假设x和y是之前定义好的数据数组
# x = np.array([...])
# y = np.array([...])

# 确保x和y是一维数组
x = x.ravel()
y = y.ravel()

# 计算Pearson相关系数和p值
print("测试集数据：")
corr, p_value = pearsonr(x, y)
print(f"Pearson R: {corr}")
print(f"pval: {p_value}")

RMSE: 0.03190902344624195
测试集数据：
Pearson R: 0.90177359784837
pval: 0.0
